# 🤝 Part 2 — The Refactoring Team (Multi-Agent Workflow)
### Workshop 4 · LLMA4SE Summer School 2026

---

**⏱ Time:** ~70 min &nbsp;·&nbsp; **Needs:** T4 GPU runtime (`Runtime → Change runtime type`)

In Part 1 one agent *found* problems. Now we build a **cooperating team** that *fixes* them — and, crucially, *proves* the fix is safe:

```
        ┌──────────┐   findings   ┌─────────────┐   candidate   ┌────────────┐
 code ─►│ AUDITOR  │─────────────►│ REFACTORER  │──────────────►│ QA VERIFIER│
        │ (Part 1) │              │ rewrites the │    code       │ syntax ✚   │
        └──────────┘              │    module    │               │ tests ✚    │
             ▲                    └─────────────┘               │ metrics    │
             │                                                   └─────┬──────┘
             │            ✅ accept  /  ❌ reject & retry              │
             └────────────────────── ORCHESTRATOR ◄────────────────────┘
```

**The big idea of Part 2:**

> 🛡️ **Generation is cheap; verification is everything.** A small LLM + a strict QA gate beats a big LLM with no gate — because the gate converts "plausible-looking code" into "provably behaviour-preserving code".

### 📚 Concepts you'll meet
- **Blackboard / shared-state pattern** — how agents communicate
- **Verification gates** — syntax → tests → metrics, in that order (cheapest first)
- **Iteration budgets** — how to stop agents looping forever
- **Agentic failure patterns** — and the mitigations that tame them


## 2.0 · Setup (run me — ~3 min)

This cell makes Part 2 **standalone**: it reinstalls packages, recreates the target project from Part 1, and reloads the model. Skip nothing, even if you just finished Part 1 in the same runtime (re-running is harmless).

In [ ]:
%pip install -q transformers accelerate radon pylint code-quality-analyzer pytest
%pip install -q git+https://github.com/KarthikShivasankar/ml_smells_detector.git
print("✅ packages ready")

In [ ]:
# Recreate the smelly module from Part 1
with open('inventory.py', 'w') as f:
    f.write('''"""inventory.py -- Order processing for a small e-commerce shop.

This module works correctly (all tests pass!) but it is deliberately
full of code smells. Your agents will find and fix them.
"""

def helper_unused(x):          # SMELL: dead code -- never called anywhere
    return x * 2


class InventoryManager:
    """Manages stock and processes customer orders."""

    def __init__(self, items=[]):              # SMELL: mutable default argument
        self.items = {}
        for name, price, qty in items:
            self.items[name] = {"price": price, "qty": qty}
        self.log = []

    def add_item(self, name, price, qty, category, supplier, discount, taxable):
        # SMELL: long parameter list (7 params, most unused)
        self.items[name] = {"price": price, "qty": qty}
        return True

    def process_order(self, order):
        # SMELL: long method, deep nesting, magic numbers, duplication
        total = 0.0
        status = "ok"
        for name, qty in order:
            if name in self.items:
                if self.items[name]["qty"] >= qty:
                    if qty > 0:
                        price = self.items[name]["price"]
                        subtotal = price * qty
                        if subtotal > 100:                      # magic number
                            subtotal = subtotal - subtotal * 0.05   # magic number
                        if qty > 10:                            # magic number
                            subtotal = subtotal - subtotal * 0.02   # magic number
                        total = total + subtotal
                        self.items[name]["qty"] = self.items[name]["qty"] - qty
                        self.log.append("sold " + name)
                    else:
                        status = "invalid_qty"
                else:
                    status = "insufficient_stock"
            else:
                status = "unknown_item"
        total = total + total * 0.25            # magic number (VAT)
        return {"total": round(total, 2), "status": status}

    def refund_order(self, order):
        # SMELL: duplicated logic (mirror of process_order maths)
        total = 0.0
        for name, qty in order:
            if name in self.items:
                price = self.items[name]["price"]
                subtotal = price * qty
                if subtotal > 100:                              # magic number again
                    subtotal = subtotal - subtotal * 0.05
                if qty > 10:
                    subtotal = subtotal - subtotal * 0.02
                total = total + subtotal
                self.items[name]["qty"] = self.items[name]["qty"] + qty
        total = total + total * 0.25
        return {"total": round(total, 2), "status": "refunded"}

    def get_stock(self, name):
        if name in self.items:
            return self.items[name]["qty"]
        return 0
''')
with open('test_inventory.py', 'w') as f:
    f.write('''"""test_inventory.py -- Behaviour-preserving safety net.

These tests define the PUBLIC CONTRACT of the module. Any refactoring
your agents perform MUST keep every one of these green.
"""
import pytest
from inventory import InventoryManager


@pytest.fixture
def mgr():
    return InventoryManager([("widget", 10.0, 100), ("gizmo", 25.0, 5)])


def test_simple_order(mgr):
    result = mgr.process_order([("widget", 2)])
    assert result["status"] == "ok"
    assert result["total"] == 25.0          # 20 + 25% VAT


def test_bulk_discount_applied(mgr):
    # 20 widgets = 200 -> -5% (>100) -> -2% (>10 units) -> +25% VAT
    result = mgr.process_order([("widget", 20)])
    assert result["total"] == 232.75


def test_stock_is_decremented(mgr):
    mgr.process_order([("widget", 2)])
    assert mgr.get_stock("widget") == 98


def test_insufficient_stock(mgr):
    result = mgr.process_order([("gizmo", 99)])
    assert result["status"] == "insufficient_stock"


def test_unknown_item(mgr):
    result = mgr.process_order([("nonexistent", 1)])
    assert result["status"] == "unknown_item"


def test_refund_restores_stock(mgr):
    mgr.process_order([("widget", 2)])
    mgr.refund_order([("widget", 2)])
    assert mgr.get_stock("widget") == 100


def test_no_shared_state_between_instances():
    a = InventoryManager()
    b = InventoryManager()
    a.items["x"] = {"price": 1, "qty": 1}
    assert "x" not in b.items or a.items is not b.items
''')
print('✅ target project recreated')
!python -m pytest test_inventory.py -q

In [ ]:
# Recreate the ML project from Part 1 (used in Exercise 2D)
import os, subprocess, pathlib
os.makedirs('ml_project', exist_ok=True)
with open('ml_project/train_model.py', 'w') as f:
    f.write('''"""train_model.py -- Churn-prediction training script.

It trains fine... but is it reproducible? Is it healthy ML code?
Your ML Auditor agent (powered by MLScent) will tell you.
"""
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


def load_data(path):
    df = pd.read_csv(path)
    for i in range(len(df)):                      # pandas: unnecessary iteration
        if df["age"][i] == np.nan:                # numpy: NaN equality (always False!)
            df["age"][i] = 0                      # pandas: chain indexing
    return df


def train():
    df = load_data("churn.csv")
    X = df.drop("label", axis=1).values
    y = df["label"].values
    # sklearn: no feature scaling, no pipeline, no random_state
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33)

    model = nn.Sequential(nn.Linear(X.shape[1], 64), nn.ReLU(), nn.Linear(64, 2))
    opt = torch.optim.Adam(model.parameters(), lr=0.003)   # hardcoded hyperparams
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(100):                      # no early stopping, no checkpoints
        out = model(torch.tensor(X_train, dtype=torch.float32))
        loss = loss_fn(out, torch.tensor(y_train))
        loss.backward()                           # pytorch: missing opt.zero_grad()
        opt.step()

    preds = model(torch.tensor(X_test, dtype=torch.float32)).argmax(1).numpy()
    print("accuracy:", accuracy_score(y_test, preds))   # over-reliance on accuracy
    # no torch.manual_seed / np.random.seed anywhere -> unreproducible


if __name__ == "__main__":
    train()
''')

def run_mlscent(project_dir: str = "ml_project") -> str:
    """MLScent: 76 ML-specific anti-pattern detectors (Shivashankar, CAIN 2025).
    Findings land in output/analysis_report.txt, so we run the CLI then read it."""
    subprocess.run(["ml_smell_detector", "analyze", project_dir],
                   capture_output=True, text=True, timeout=300)
    report = pathlib.Path("output/analysis_report.txt")
    return report.read_text()[:3500] if report.exists() else "MLScent produced no report"
print('✅ ml_project recreated')

In [ ]:
# ============================================================
#  Load a small open-weights code LLM (fits free Colab T4 GPU)
# ============================================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"   # ~3 GB in fp16
# Slower machine / CPU-only fallback:
# MODEL_NAME = "Qwen/Qwen2.5-Coder-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading {MODEL_NAME} on {device} ...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",
)
print("Model loaded ✔")


def llm(user_prompt: str, system_prompt: str = "You are a helpful assistant.",
        max_new_tokens: int = 1024, temperature: float = 0.2) -> str:
    """One call to our local LLM. Every agent in this workshop uses this."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    ).strip()


In [ ]:
# Compact version of Part 1's auditor (tools + JSON-extracting audit function)
import subprocess, json, re, ast, pathlib

def run_radon(path):
    cc = subprocess.run(["radon", "cc", "-s", path], capture_output=True, text=True).stdout
    mi = subprocess.run(["radon", "mi", "-s", path], capture_output=True, text=True).stdout
    return f"COMPLEXITY:\n{cc}\nMAINTAINABILITY:\n{mi}"

def run_pylint(path):
    raw = subprocess.run(["pylint", path, "--output-format=json",
                          "--disable=C0114,C0115,C0116"], capture_output=True, text=True).stdout
    try:
        return "\n".join(f"L{i['line']}: [{i['symbol']}] {i['message']}"
                          for i in json.loads(raw)[:15]) or "no findings"
    except json.JSONDecodeError:
        return raw[:1500]

def extract_json(text):
    m = re.search(r"\[.*\]|\{.*\}", text, re.DOTALL)
    if not m:
        raise ValueError(f"No JSON in: {text[:200]}")
    return json.loads(m.group(0))

AUDITOR_PROMPT = """You are a meticulous senior code reviewer. Given a Python file
and static-analysis evidence, list the most important code smells.
Reply ONLY with a JSON array of objects:
{"smell": str, "location": str, "severity": "high|medium|low",
 "why": str, "fix": str}. Max 6 findings, most severe first."""

def audit(path: str) -> list[dict]:
    evidence = f"=== radon ===\n{run_radon(path)}\n\n=== pylint ===\n{run_pylint(path)}"
    source = open(path).read()
    raw = llm(f"SOURCE ({path}):\n```python\n{source}\n```\n\nEVIDENCE:\n{evidence}\n\nJSON findings now.",
              system_prompt=AUDITOR_PROMPT, max_new_tokens=800, temperature=0.1)
    return extract_json(raw)

print("✅ auditor ready")

## 2.1 · How do agents talk? The shared-state ("blackboard") pattern

Multi-agent systems need a communication protocol. The two dominant designs:

| Pattern | How it works | Used by |
|---|---|---|
| **Message passing** | agents send addressed messages to each other | AutoGen-style "conversations" |
| **Blackboard / shared state** | agents read & write one shared state object; an orchestrator decides who acts next | LangGraph-style "graphs" ✅ *(what we build)* |

We use the blackboard: it's simpler to reason about, trivially loggable, and every write is **audit-trailed** — you can replay exactly what each agent saw and did. That trace is your best debugging tool when agents misbehave.

In [ ]:
from dataclasses import dataclass, field
import datetime

@dataclass
class WorkflowState:
    """The blackboard: one source of truth all agents read/write."""
    source_path: str
    original_code: str = ""
    candidate_code: str = ""      # the refactorer's latest proposal
    findings: list = field(default_factory=list)
    verdict: dict = field(default_factory=dict)
    accepted: bool = False
    iteration: int = 0
    history: list = field(default_factory=list)   # the audit trail

    def record(self, agent: str, event: str, detail: str = ""):
        stamp = datetime.datetime.now().strftime("%H:%M:%S")
        self.history.append(f"[{stamp}] {agent:12} | {event:22} | {detail}")
        print(self.history[-1])

state = WorkflowState(source_path="inventory.py")
state.original_code = open(state.source_path).read()
state.record("system", "state initialised", f"{len(state.original_code)} chars of smelly code")

## 2.2 · Agent 2: the **Refactoring Agent** 🔧

Its contract: *given the source + the auditor's findings, produce a complete rewritten module that fixes the smells **without changing behaviour***.

Three guardrails make a 1.5B model usable here — read the prompt carefully, each line exists for a reason:

1. **Explicit invariants** — "keep the public API", "same return values" (otherwise the model 'helpfully' redesigns your API).
2. **Fenced-code-only output** — so we can extract the module mechanically.
3. **Low temperature (0.1)** — refactoring wants precision, not creativity.

In [ ]:
REFACTORER_PROMPT = """You are an expert Python refactoring engineer.
Rewrite the ENTIRE module to fix the reported code smells.

HARD RULES — violating any of these makes your output worthless:
1. Public API must not change: class InventoryManager with methods
   __init__(items=None), add_item, process_order, refund_order, get_stock.
2. Behaviour must be IDENTICAL: same return values for the same inputs,
   including all totals, discounts, tax and status strings.
3. Use ONLY the Python standard library. Do NOT invent helper packages.
4. Replace magic numbers with named module-level constants.
5. Extract duplicated pricing logic into one private helper method.
6. Fix the mutable default argument. Remove dead code.

Output ONLY one fenced python code block containing the complete module.
No explanations before or after."""

def extract_code_block(text: str) -> str:
    """Pull the (last) fenced python block out of an LLM reply."""
    blocks = re.findall(r"```(?:python)?\s*\n(.*?)```", text, re.DOTALL)
    if not blocks:
        raise ValueError("Refactorer produced no code block")
    return blocks[-1].strip() + "\n"

def refactor(state: WorkflowState) -> WorkflowState:
    state.record("refactorer", "thinking", f"{len(state.findings)} findings to fix")
    findings_txt = json.dumps(state.findings, indent=1)
    reply = llm(
        f"MODULE TO REFACTOR:\n```python\n{state.original_code}\n```\n\n"
        f"AUDITOR FINDINGS:\n{findings_txt}\n\nRewrite the full module now.",
        system_prompt=REFACTORER_PROMPT, max_new_tokens=1600, temperature=0.1,
    )
    state.candidate_code = extract_code_block(reply)
    state.record("refactorer", "candidate produced", f"{len(state.candidate_code)} chars")
    return state

## 2.3 · Agent 3: the **QA Verification Agent** 🛡️

This agent is the *only* reason we can trust anything the refactorer produces. It runs three gates, **cheapest first** — a classic engineering pattern (fail fast, spend compute only on survivors):

| Gate | Cost | Catches |
|---|---|---|
| 1 · `ast.parse` syntax check | ~1 ms | truncated / mangled output |
| 2 · full `pytest` suite in a sandbox | ~1 s | behavioural regressions |
| 3 · `radon` complexity comparison | ~0.1 s | "fixes" that made the code *worse* |

Note what's **not** here: no LLM. The QA agent is deliberately deterministic — you never want the fox auditing the henhouse.

In [ ]:
import shutil, tempfile, os

def avg_complexity(path: str) -> float:
    """Average cyclomatic complexity of a file (via radon's JSON output)."""
    raw = subprocess.run(["radon", "cc", "-j", path], capture_output=True, text=True).stdout
    data = json.loads(raw)
    scores = [b["complexity"] for blocks in data.values() for b in blocks]
    return sum(scores) / len(scores) if scores else 0.0

def qa_verify(state: WorkflowState) -> WorkflowState:
    verdict = {"syntax": False, "tests": False, "improved": False, "notes": []}

    # ---- Gate 1: does it even parse? ----
    try:
        ast.parse(state.candidate_code)
        verdict["syntax"] = True
    except SyntaxError as e:
        verdict["notes"].append(f"SyntaxError: {e}")
        state.verdict = verdict
        state.record("qa", "❌ REJECTED at gate 1", str(e)[:60])
        return state

    # ---- Gate 2: behaviour preserved? (run tests in a sandbox) ----
    with tempfile.TemporaryDirectory() as sandbox:
        open(os.path.join(sandbox, "inventory.py"), "w").write(state.candidate_code)
        shutil.copy("test_inventory.py", sandbox)
        r = subprocess.run(["python", "-m", "pytest", "test_inventory.py", "-q"],
                           cwd=sandbox, capture_output=True, text=True, timeout=120)
        verdict["tests"] = (r.returncode == 0)
        if not verdict["tests"]:
            verdict["notes"].append("pytest output:\n" + r.stdout[-800:])
            state.verdict = verdict
            state.record("qa", "❌ REJECTED at gate 2", "tests failed")
            return state

        # ---- Gate 3: is it actually better? ----
        cc_before = avg_complexity(state.source_path)
        cc_after  = avg_complexity(os.path.join(sandbox, "inventory.py"))
        verdict["cc_before"], verdict["cc_after"] = round(cc_before, 2), round(cc_after, 2)
        verdict["improved"] = cc_after <= cc_before
        if not verdict["improved"]:
            verdict["notes"].append(f"complexity got WORSE: {cc_before} → {cc_after}")

    state.verdict = verdict
    ok = verdict["syntax"] and verdict["tests"] and verdict["improved"]
    state.record("qa", "✅ ACCEPTED" if ok else "❌ REJECTED at gate 3",
                 f"CC {verdict.get('cc_before','?')} → {verdict.get('cc_after','?')}")
    return state

## 2.4 · The **Orchestrator**: closing the loop 🔁

The orchestrator wires the team together and adds the two properties every agentic system needs:

- **A retry loop** — if QA rejects, the refactorer gets another shot **with the failure notes fed back** (this feedback is what makes iteration 2 smarter than iteration 1).
- **A budget** (`max_iterations`) — the difference between an agent workflow and an infinite money-burning loop.

In [ ]:
def run_workflow(source_path: str, max_iterations: int = 3) -> WorkflowState:
    state = WorkflowState(source_path=source_path)
    state.original_code = open(source_path).read()

    state.record("orchestrator", "▶ audit phase")
    state.findings = audit(source_path)
    state.record("auditor", "findings", ", ".join(f["smell"] for f in state.findings)[:80])

    feedback = ""
    for state.iteration in range(1, max_iterations + 1):
        state.record("orchestrator", f"▶ iteration {state.iteration}/{max_iterations}")

        if feedback:   # feed QA failure notes back to the refactorer
            state.findings = state.findings + [
                {"smell": "PREVIOUS ATTEMPT FAILED QA", "why": feedback[:400],
                 "fix": "produce a corrected complete module"}]

        state = refactor(state)
        state = qa_verify(state)

        v = state.verdict
        if v.get("syntax") and v.get("tests") and v.get("improved"):
            state.accepted = True
            break
        feedback = " | ".join(v.get("notes", []))[:400]

    state.record("orchestrator",
                 "🏁 DONE — patch " + ("ACCEPTED" if state.accepted else "NOT accepted"),
                 f"after {state.iteration} iteration(s)")
    return state

state = run_workflow("inventory.py", max_iterations=3)

### 🔍 Inspect the result

> ⚠️ **It's fine (and instructive!) if some iterations were rejected.** A 1.5B model *will* sometimes break behaviour — and your QA gate just caught it in front of your eyes. That catch **is** the lesson: the gate, not the model, is what makes the system trustworthy.

Let's look at the diff and the before/after metrics.

In [ ]:
import difflib

if state.accepted:
    diff = difflib.unified_diff(
        state.original_code.splitlines(keepends=True),
        state.candidate_code.splitlines(keepends=True),
        fromfile="inventory.py (before)", tofile="inventory.py (after)")
    print("".join(diff))
else:
    print("No accepted patch this run — re-run the workflow cell (sampling varies),")
    print("or raise max_iterations. Meanwhile inspect why QA said no:")
    print(json.dumps(state.verdict, indent=2))

In [ ]:
# Before / after scoreboard
mi = lambda p: subprocess.run(["radon", "mi", "-s", p], capture_output=True, text=True).stdout.strip()
print("METRIC                BEFORE                       AFTER")
print("-" * 70)
if state.accepted:
    open("inventory_refactored.py", "w").write(state.candidate_code)
    print(f"avg complexity        {state.verdict['cc_before']:<28} {state.verdict['cc_after']}")
    print(f"maintainability       {mi('inventory.py'):<28.28} {mi('inventory_refactored.py'):<.28}")
    print(f"tests                 7 passed                     7 passed  ✅ behaviour preserved")
else:
    print("(accept a patch first)")

In [ ]:
# The audit trail: every decision, replayable. THIS is how you debug agents.
print("\n".join(state.history))

## 2.5 · 🔥 The Failure-Pattern Lab

Time to break things on purpose. Agentic systems fail in *recurring, nameable* ways. Knowing the names is half the defence.

| # | Failure pattern | What it looks like | Mitigation (which we built!) |
|---|---|---|---|
| 1 | **Plausible-but-wrong code** | patch looks clean, changes behaviour | test-suite gate (Gate 2) |
| 2 | **Hallucinated APIs** | `import superfixlib` | stdlib-only rule + sandbox execution |
| 3 | **Truncated / mangled output** | half a module | syntax gate (Gate 1) |
| 4 | **Infinite retry loops** | agent never converges, bill explodes | iteration budget |
| 5 | **Goal drift** | "while I'm here, I redesigned your API" | explicit invariants in prompt + API check |
| 6 | **Metric gaming** | deletes code to lower complexity | multiple gates: tests AND metrics |

**Demo — watch Gate 2 earn its salary.** We inject a *plausible-looking sabotage*: a patch that renames things nicely but silently changes the VAT rate from 25% to 20%. A human skimming the diff would likely approve it…

In [ ]:
sabotage = state.original_code.replace("total * 0.25", "total * 0.20")  # 🕵️ subtle!

evil_state = WorkflowState(source_path="inventory.py")
evil_state.original_code = state.original_code
evil_state.candidate_code = sabotage
evil_state = qa_verify(evil_state)

print("\nQA verdict on the sabotaged patch:")
print(json.dumps({k: v for k, v in evil_state.verdict.items() if k != "notes"}, indent=2))
print("\nFailure notes:\n", "\n".join(evil_state.verdict["notes"])[:600])

☝️ The tests caught in ~1 second what code review might miss. Now flip it around:

> 💬 **Discuss (2 min):** Gate 2 is only as strong as the test suite. What sabotage would slip through *our* seven tests? (Hint: is `add_item`'s discount parameter tested? What about float edge cases?) — This is why *test coverage* is itself a technical-debt dimension.

## 2.6 · ✍️ Exercise 2 (15 min) — extend the team

Pick **one** (or more, if you're flying):

**A. Docstring Agent (easier).** Add a fourth agent that takes the *accepted* code and adds Google-style docstrings to every method. It must run **after** QA acceptance, and its output must pass `qa_verify` again before being kept. *(Why after? Because even adding comments can break code — trust nothing.)*

**B. Stricter Gate 3 (medium).** Extend `qa_verify` so the Maintainability Index must also *improve* (`radon mi -j` gives JSON). Does the acceptance rate drop? Is that good or bad?

**C. Budget accounting (harder).** Add `tokens_spent` to `WorkflowState`, count generated tokens per call, and make the orchestrator abort when a token budget is exceeded. Congratulations — you've built cost governance, the #1 requirement for production agents.

**D. ML refactoring team (harder, for ML folks).** Point the workflow at `ml_project/train_model.py` from Part 1. Catch: it has **no test suite** — so gate 2 can't be pytest. Design a replacement gate: e.g. *the MLScent smell count must strictly decrease* (re-run `run_mlscent()` on the patched file and compare `Smell Counts`), plus `ast.parse` for syntax. What behaviour guarantees do you lose without tests? What could a smell-count gate be fooled by?

In [ ]:
# 🖊️ Your Exercise 2 workspace

# Skeleton for option A:
DOCSTRING_PROMPT = """You are a documentation engineer. Add concise Google-style
docstrings to every class and method. Change NOTHING else — not one identifier,
not one expression. Output ONLY one fenced python code block."""

def document(state):
    # TODO: call llm() with DOCSTRING_PROMPT + state.candidate_code,
    #       extract_code_block(), put result back into state.candidate_code,
    #       then re-run qa_verify(state) and only keep it if accepted.
    pass

<details><summary>💡 Solution sketch (option A)</summary>

```python
def document(state):
    reply = llm(f"```python\n{state.candidate_code}\n```",
                system_prompt=DOCSTRING_PROMPT, max_new_tokens=1600, temperature=0.1)
    proposal = extract_code_block(reply)
    trial = WorkflowState(source_path=state.source_path)
    trial.original_code = state.original_code
    trial.candidate_code = proposal
    trial = qa_verify(trial)
    if trial.verdict.get("tests"):
        state.candidate_code = proposal
        state.record("documenter", "docstrings added & verified")
    else:
        state.record("documenter", "proposal rejected by QA — keeping previous code")
    return state

if state.accepted:
    state = document(state)
```
</details>

## 2.7 · Checkpoint ✅

1. Why is the QA agent deliberately **not** an LLM?
2. Order the three gates by cost, and explain why order matters.
3. Name three agentic failure patterns and their mitigations from memory.

<details><summary>Answers</summary>

1. Verification must be trustworthy and deterministic; an LLM verifier can be fooled by (or share blind spots with) the LLM generator.
2. syntax (ms) → tests (s) → metrics; fail fast so expensive checks only run on plausible candidates.
3. See the table in 2.5 — e.g. hallucinated APIs → sandboxed execution; plausible-but-wrong → test gate; runaway loops → iteration budget.
</details>

➡️ **Part 3:** we zoom out from one file to the **portfolio view** — classifying and prioritising technical debt, and assembling the full pipeline.